# 출력 파서

In [2]:
from langchain_core.output_parsers import JsonOutputParser 

# Json 출력 파서 불러오기 
parser = JsonOutputParser()
instructions = parser.get_format_instructions()
print(instructions)

Return a JSON object.


In [7]:
# ai_response = "{'이름' : '김철수', '나이' : 30}"
ai_response = '{"이름" : "김철수", "나이" : 30}'
parsed_response = parser.parse(ai_response)
print(parser)

#### 프롬프트와 함께 파싱

In [ ]:
from langchain_classic.output_parsers import RetryWithErrorOutputParser
from langchain_core.output_parsers import JsonOutputParser 
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gemma-3-4b-it",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

parser = RetryWithErrorOutputParser.from_llm(parser=JsonOutputParser(), llm=llm)


In [ ]:
question = "가장 큰 대륙은"
ai_wrong_response = "아시아입니다"
ai_correct_response = '{"answer" : "아시아입니다"}'

try:
    result = parser.parse_with_prompt(ai_correct_response, question)
    print(result)
except Exception as e: 
    print(f"오류 발생 : {e}") 

{'answer': '아시아입니다'}


#### PydanticOutputParser

In [17]:
from langchain_core.output_parsers import PydanticOutputParser 
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI 
from pydantic import BaseModel, Field, model_validator 

llm = ChatOpenAI(
    model="gemma-3-4b-it",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    temperature=0.0
)

In [20]:
class FinancialAdvice(BaseModel):
    setup : str = Field(description="금융 조언 상활을 설정하기 위한 질문")
    advice : str = Field(description="질문을 해결하기 위한 금융 답변")
    
    @model_validator(mode = "before")
    @classmethod 
    def question_end_with_question_mark(cls, values: dict) -> dict: 
        setup = values.get("setup" , "")
        if not setup.endswith("?"):
            raise ValueError("잘못된 질문 형식입니다! 질문은 '?'로 끝나야 합니다")
        return values

In [27]:
parser = PydanticOutputParser(pydantic_object=FinancialAdvice)
prompt = PromptTemplate(
    template="""다음 금융 관련 요청에 답변해주세요.

    반드시 JSON만 출력하세요.;
    setup은 반드시 하나의 질문문이어야 합니다.
    setup은 반드시 물음표(?)로 끝나야 합니다.
    setup 끝 문자는 마침표(.)가 아니라 반드시 ? 여야 합니다.
    advice는 setup 질문에 대한 답변입니다.

    나쁜 예:
    {{"setup": "부동산 조언을 부탁드립니다.", "advice": "..."}}

    좋은 예:
    {{"setup": "부동산 투자를 시작하기 전에 어떤 재무 상태를 먼저 점검해야 하나요?", "advice": "..."}}
    {format_instruction} \n
    advice 값은 setup 질문에 대한 답변입니다. \n
    
    질문: {query}\n""",
    input_variables=["query"],
    partial_variables={"format_instruction": parser.get_format_instructions()}
)

chain = prompt | llm | parser
try:
    result = chain.invoke({"query" : "부동산에 관련하여 금융조언을 받을 수 있게 질문"})
    print(result)
except Exception as e:
    print(f"오류 발생 {e}")

setup='현재 고금리 및 불확실한 시장 상황에서, 처음 부동산 투자를 고려하는 사람에게 가장 리스크를 줄이면서도 성공적인 진입을 위한 전략은 무엇인가요?' advice="가장 먼저 본인의 재정 상태를 점검해야 합니다. 투자금의 출처, 현금 흐름 안정성, 그리고 대출을 받을 경우 상환 능력을 객관적으로 평가하세요. 다음으로, '묻지마 투자' 대신 해당 지역의 장기적인 수요 증가 추이와 임대 수익률을 분석하여 투자 목적에 맞는 부동산 유형(주거용, 상업용 등)을 선택하는 것이 중요합니다. 시장 타이밍을 맞추려 하기보다는, 본인의 투자 목표와 자금 상황에 맞는 '나만의 속도'를 찾는 것이 성공적인 투자의 핵심입니다."


#### SimpleJsonOutputParser

In [30]:
from langchain_core.output_parsers import SimpleJsonOutputParser 

json_prompt = PromptTemplate.from_template(
    "다음 질문에 대한 답변이 포함된 JSON객체를 반환하십시오: {question}"
)
json_parser = SimpleJsonOutputParser()
json_chain = json_prompt | llm | json_parser 

response = json_chain.invoke({"question":"비트코인에 대한 짧은 설명"})
print(type(response))


<class 'dict'>


In [31]:
list(json_chain.stream({"question":"비트코인에 대한 짧은 설명"}))

[{},
 {'topic': ''},
 {'topic': '비'},
 {'topic': '비트'},
 {'topic': '비트코'},
 {'topic': '비트코인'},
 {'topic': '비트코인 ('},
 {'topic': '비트코인 (Bitcoin'},
 {'topic': '비트코인 (Bitcoin)'},
 {'topic': '비트코인 (Bitcoin)', 'description': ''},
 {'topic': '비트코인 (Bitcoin)', 'description': '비'},
 {'topic': '비트코인 (Bitcoin)', 'description': '비트'},
 {'topic': '비트코인 (Bitcoin)', 'description': '비트코'},
 {'topic': '비트코인 (Bitcoin)', 'description': '비트코인은'},
 {'topic': '비트코인 (Bitcoin)', 'description': '비트코인은 블'},
 {'topic': '비트코인 (Bitcoin)', 'description': '비트코인은 블록'},
 {'topic': '비트코인 (Bitcoin)', 'description': '비트코인은 블록체'},
 {'topic': '비트코인 (Bitcoin)', 'description': '비트코인은 블록체인'},
 {'topic': '비트코인 (Bitcoin)', 'description': '비트코인은 블록체인 기술'},
 {'topic': '비트코인 (Bitcoin)', 'description': '비트코인은 블록체인 기술을'},
 {'topic': '비트코인 (Bitcoin)', 'description': '비트코인은 블록체인 기술을 기반'},
 {'topic': '비트코인 (Bitcoin)', 'description': '비트코인은 블록체인 기술을 기반으로'},
 {'topic': '비트코인 (Bitcoin)', 'description': '비트코인은 블록체인 기술을 기반으로 하는'},
 {'topic

#### JsonOutputParser

In [33]:
from langchain_core.output_parsers import JsonOutputParser 
from langchain_core.prompts import PromptTemplate 
from pydantic import BaseModel, Field 

class FinancialAdvice(BaseModel):
    setup : str = Field(description="금융 조언 상활을 설정하기 위한 질문")
    advice : str = Field(description="질문을 해결하기 위한 답변")

parser = JsonOutputParser(pydantic_object=FinancialAdvice)
prompt = PromptTemplate.from_template(
    template= """
    다음 금융 관련 질문에 답변해주세요.\n
    {format_instructions}\n
    {query}\n
    """, 
    input_variable=["query"], 
    partial_variables={"format_instructions": parser.get_format_instructions()}   
)

chain = prompt | llm | parser

response = chain.invoke({"query": "부동산에 관련하여 금융 조언을 받을 수 있게 질문하라"})

In [34]:
response

{'setup': '현재 목돈이 생겼는데, 향후 5년 동안 안정적인 수익을 얻으면서 자산 증식을 할 수 있는 부동산 투자처(아파트 vs 상가)에 대해 조언을 구하고 싶습니다. 현재 시장 상황과 저의 투자 성향(중위험 선호)을 고려했을 때 어떤 선택이 좋을까요?',
 'advice': '현재 시장 동향, 각 부동산 유형의 리스크 및 수익률 분석을 바탕으로 저의 투자 목표에 맞는 최적의 포트폴리오 구성 및 진입 시점에 대한 전문적인 조언을 부탁드립니다.'}